In [1]:
# Importing the libraries

import pandas as pd
import json
import os


In [2]:
ledger = pd.read_csv('/content/ledger.csv')
gateway = pd.read_csv('/content/gateway.csv')

In [3]:
print(ledger.shape)
print(gateway.shape)


(10, 6)
(9, 6)


In [4]:
ledger.head()

,transaction_id,transaction_date,merchant_id,amount_usd,status,payment_method
0,R001,2026-03-01,M001,1200.0,success,UPI
1,R002,2026-03-01,M002,850.0,success,Card
2,R003,2026-03-02,M001,500.0,success,Wallet
3,R004,2026-03-02,M003,2100.0,success,Card
4,R005,2026-03-03,M004,7200.0,success,Card


In [5]:
gateway.head()

,transaction_id,transaction_date,merchant_id,amount_usd,status,payment_method
0,R001,2026-03-01,M001,1200.0,success,UPI
1,R002,2026-03-01,M002,900.0,success,Card
2,R003,2026-03-02,M001,500.0,success,Wallet
3,R005,2026-03-03,M004,7200.0,failed,Card
4,R006,2026-03-03,M002,950.0,success,UPI


In [6]:
ledger.describe()

,amount_usd
count,10.000000
mean,2334.000000
std,2093.615692
min,500.000000
25%,875.000000
50%,1650.000000
75%,3100.000000
max,7200.000000


In [7]:
gateway.describe()

,amount_usd
count,9.000000
mean,2283.333333
std,2225.421308
min,500.000000
25%,900.000000
50%,1200.000000
75%,3300.000000
max,7200.000000


In [8]:
# Cheking nulls

print("Ledger Nulls:\n", ledger.isnull().sum())
print("Gateway Nulls:\n", gateway.isnull().sum())

Ledger Nulls:
 transaction_id      0
transaction_date    0
merchant_id         0
amount_usd          0
status              0
payment_method      0
dtype: int64
Gateway Nulls:
 transaction_id      0
transaction_date    0
merchant_id         0
amount_usd          0
status              0
payment_method      0
dtype: int64


In [9]:
# Checking for Duplicates

ledger_dup = ledger[ledger.duplicated(subset=['transaction_id'], keep=False)]
gateway_dup = gateway[gateway.duplicated(subset=['transaction_id'], keep=False)]

print("Ledger Duplicates:\n", ledger_dup)
print("Gateway Duplicates:\n", gateway_dup)

Ledger Duplicates:
 Empty DataFrame
Columns: [transaction_id, transaction_date, merchant_id, amount_usd, status, payment_method]
Index: []
Gateway Duplicates:
 Empty DataFrame
Columns: [transaction_id, transaction_date, merchant_id, amount_usd, status, payment_method]
Index: []


In [10]:
recon_df = pd.merge(
    ledger,
    gateway,
    on='transaction_id',
    how='outer',
    suffixes=('_ledger', '_gateway')
)

In [11]:
# Checks for the missing in gateway, value is in ledger but missing in gateway
missing_in_gateway = recon_df[recon_df['amount_usd_gateway'].isna()].copy()
print("Missing in Gateway:\n", missing_in_gateway)

Missing in Gateway:
   transaction_id transaction_date_ledger merchant_id_ledger  \
3           R004              2026-03-02               M003   
9           R010              2026-03-05               M004   

   amount_usd_ledger status_ledger payment_method_ledger  \
3             2100.0       success                  Card   
9             2500.0       success                Wallet   

  transaction_date_gateway merchant_id_gateway  amount_usd_gateway  \
3                      NaN                 NaN                 NaN   
9                      NaN                 NaN                 NaN   

  status_gateway payment_method_gateway  
3            NaN                    NaN  
9            NaN                    NaN  


In [12]:
# Checks for the missing in ledger, value present in the gateway but missing in ledger

missing_in_ledger = recon_df[recon_df['amount_usd_ledger'].isna()].copy()
print("Missing in Ledger:\n", missing_in_ledger)

Missing in Ledger:
    transaction_id transaction_date_ledger merchant_id_ledger  \
10           R011                     NaN                NaN   

    amount_usd_ledger status_ledger payment_method_ledger  \
10                NaN           NaN                   NaN   

   transaction_date_gateway merchant_id_gateway  amount_usd_gateway  \
10               2026-03-05                M003              1800.0   

   status_gateway payment_method_gateway  
10        success                   Card  


In [13]:
# Checking for the mismatches

amount_mismatches = recon_df[
    recon_df['amount_usd_ledger'].notna() &
    recon_df['amount_usd_gateway'].notna() &
    (recon_df['amount_usd_ledger'] != recon_df['amount_usd_gateway'])
].copy()

print("Amount Mismatches:\n", amount_mismatches)

Amount Mismatches:
   transaction_id transaction_date_ledger merchant_id_ledger  \
1           R002              2026-03-01               M002   
7           R008              2026-03-04               M001   

   amount_usd_ledger status_ledger payment_method_ledger  \
1              850.0       success                  Card   
7              640.0       success                  Card   

  transaction_date_gateway merchant_id_gateway  amount_usd_gateway  \
1               2026-03-01                M002               900.0   
7               2026-03-04                M001               600.0   

  status_gateway payment_method_gateway  
1        success                   Card  
7        success                   Card  


In [14]:
# Checks for the Status Mismatches

status_mismatches = recon_df[
    recon_df['status_ledger'].notna() &
    recon_df['status_gateway'].notna() &
    (recon_df['status_ledger'].str.strip().str.lower() != recon_df['status_gateway'].str.strip().str.lower())
].copy()

print("Status Mismatches",status_mismatches)

Status Mismatches   transaction_id transaction_date_ledger merchant_id_ledger  \
4           R005              2026-03-03               M004   

   amount_usd_ledger status_ledger payment_method_ledger  \
4             7200.0       success                  Card   

  transaction_date_gateway merchant_id_gateway  amount_usd_gateway  \
4               2026-03-03                M004              7200.0   

  status_gateway payment_method_gateway  
4         failed                   Card  


In [15]:
def get_recon_status(row):
    if pd.isna(row['amount_usd_gateway']):
        return 'Missing in Gateway'
    if pd.isna(row['amount_usd_ledger']):
        return 'Missing in Ledger'
    if row['amount_usd_ledger'] != row['amount_usd_gateway']:
        return 'Amount Mismatch'
    if row['status_ledger'].strip().lower() != row['status_gateway'].strip().lower():
        return 'Status Mismatch'
    return 'Reconciled'

# Correcting indentation so this runs after the function is defined
recon_df['reconciliation_status'] = recon_df.apply(get_recon_status, axis=1)

In [16]:
summary_metrics = {
    "total_ledger_rows": int(len(ledger)),
    "total_gateway_rows": int(len(gateway)),
    "missing_in_gateway_count": int(len(missing_in_gateway)),
    "missing_in_ledger_count": int(len(missing_in_ledger)),
    "amount_mismatch_count": int(len(amount_mismatches)),
    "status_mismatch_count": int(len(status_mismatches)),
    "fully_reconciled_count": int(len(recon_df[recon_df['reconciliation_status'] == 'Reconciled']))
}

summary_metrics["reconciliation_issue_count"] = (
    summary_metrics["missing_in_gateway_count"] +
    summary_metrics["missing_in_ledger_count"] +
    summary_metrics["amount_mismatch_count"] +
    summary_metrics["status_mismatch_count"]
)

summary_metrics["ledger_total_amount"] = float(ledger['amount_usd'].sum())
summary_metrics["gateway_total_amount"] = float(gateway['amount_usd'].sum())

# Calculate amount at risk
amount_at_risk = 0.0
amount_at_risk += recon_df[recon_df['reconciliation_status'] == 'Missing in Gateway']['amount_usd_ledger'].sum()
amount_at_risk += recon_df[recon_df['reconciliation_status'] == 'Missing in Ledger']['amount_usd_gateway'].sum()
amount_at_risk += abs(recon_df[recon_df['reconciliation_status'] == 'Amount Mismatch']['amount_usd_ledger'] - recon_df[recon_df['reconciliation_status'] == 'Amount Mismatch']['amount_usd_gateway']).sum()
amount_at_risk += recon_df[recon_df['reconciliation_status'] == 'Status Mismatch']['amount_usd_ledger'].sum() # Assuming full amount is at risk for status mismatches

summary_metrics["amount_at_risk"] = float(amount_at_risk)

# Print each summary metric on a new line
for key, value in summary_metrics.items():
    print(f"{key}: {value}")

total_ledger_rows: 10
total_gateway_rows: 9
missing_in_gateway_count: 2
missing_in_ledger_count: 1
amount_mismatch_count: 2
status_mismatch_count: 1
fully_reconciled_count: 5
reconciliation_issue_count: 6
ledger_total_amount: 23340.0
gateway_total_amount: 20550.0
amount_at_risk: 13690.0


In [17]:
# Generating the CSV files

missing_in_gateway.to_csv('missing_in_gateway.csv', index=False)
missing_in_ledger.to_csv('missing_in_ledger.csv', index=False)
amount_mismatches.to_csv('amount_mismatches.csv', index=False)
status_mismatches.to_csv('status_mismatches.csv', index=False)
recon_df.to_csv('reconciliation_report.csv', index=False)


print('All reconciliation reports and summary metrics have been saved.')
with open('summary_metrics.json', 'w') as f:
    json.dump(summary_metrics, f, indent=4)

print('All reconciliation reports and summary metrics have been saved.')

All reconciliation reports and summary metrics have been saved.
All reconciliation reports and summary metrics have been saved.


In [18]:
import json

with open('summary_metrics.json', 'r') as f:
    summary_data = json.load(f)
print(json.dumps(summary_data, indent=4))

{
    "total_ledger_rows": 10,
    "total_gateway_rows": 9,
    "missing_in_gateway_count": 2,
    "missing_in_ledger_count": 1,
    "amount_mismatch_count": 2,
    "status_mismatch_count": 1,
    "fully_reconciled_count": 5,
    "reconciliation_issue_count": 6,
    "ledger_total_amount": 23340.0,
    "gateway_total_amount": 20550.0,
    "amount_at_risk": 13690.0
}


# JSON Normalization Task

In [19]:
# Load and normalize JSON data
with open('/content/api_response_sample.json', 'r') as f:
    api_data = json.load(f)


if isinstance(api_data, dict) and 'transactions' in api_data:
    normalized_df = pd.json_normalize(api_data['transactions'])
else:
    normalized_df = pd.json_normalize(api_data)

display(normalized_df.head())

,generated_at,source,batches
0,2026-03-07T10:00:00Z,QuickPay Settlement API,"[{'batch_id': 'B001', 'merchant': {'merchant_i..."


In [20]:
# Clean column names and convert date/time fields

# Clean Column names
normalized_df.columns = [c.replace('.', '_').lower() for c in normalized_df.columns]

# Convert date/time fields

date_cols = [col for col in normalized_df.columns if 'date' in col or 'time' in col or 'timestamp' in col]
for col in date_cols:
    normalized_df[col] = pd.to_datetime(normalized_df[col], errors='coerce')

print("Cleaned Columns:", normalized_df.columns.tolist())
display(normalized_df.info())
display(normalized_df.head())

Cleaned Columns: ['generated_at', 'source', 'batches']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1 entries, 0 to 0
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   generated_at  1 non-null      object
 1   source        1 non-null      object
 2   batches       1 non-null      object
dtypes: object(3)
memory usage: 156.0+ bytes


None

,generated_at,source,batches
0,2026-03-07T10:00:00Z,QuickPay Settlement API,"[{'batch_id': 'B001', 'merchant': {'merchant_i..."


In [21]:

df_batches = normalized_df.explode('batches').reset_index(drop=True)

# Normalize the nested batch and transaction data
# Extracting the 'batches' column which contains merchant and transaction info
transactions_list = []
for index, row in df_batches.iterrows():
    batch = row['batches']
    if isinstance(batch, dict) and 'settlements' in batch:
        # Normalize transactions in this batch
        batch_tx = pd.json_normalize(batch['settlements'])
        # Add batch-level info
        batch_tx['batch_id'] = batch.get('batch_id')
        batch_tx['merchant_id'] = batch.get('merchant', {}).get('merchant_id')
        batch_tx['merchant_name'] = batch.get('merchant', {}).get('merchant_name')
        # Add top-level metadata
        batch_tx['api_generated_at'] = row['generated_at']
        batch_tx['api_source'] = row['source']
        transactions_list.append(batch_tx)

# Combine all transactions into a single flat table
final_api_table = pd.concat(transactions_list, ignore_index=True)

# Clean columns (lowercase, replace dots)
final_api_table.columns = [c.replace('.', '_').lower() for c in final_api_table.columns]

# Reorder columns for better readability
cols = ['transaction_id', 'date', 'amount', 'currency', 'status', 'merchant_id', 'batch_id']
existing_cols = [c for c in cols if c in final_api_table.columns]
remaining_cols = [c for c in final_api_table.columns if c not in existing_cols]
final_api_table = final_api_table[existing_cols + remaining_cols]

display(final_api_table.head())
final_api_table.to_csv('api_normalized.csv', index=False)
print('\nFlat transaction table saved to api_normlized.csv')

,status,merchant_id,batch_id,settlement_id,amount_usd,processed_at,bank_name,bank_country,merchant_name,api_generated_at,api_source
0,settled,M001,B001,S001,1520.5,2026-03-07T08:10:00Z,Bank A,IN,Alpha Mart,2026-03-07T10:00:00Z,QuickPay Settlement API
1,pending,M001,B001,S002,980.0,2026-03-07T08:45:00Z,Bank A,IN,Alpha Mart,2026-03-07T10:00:00Z,QuickPay Settlement API
2,settled,M001,B001,S003,640.0,2026-03-07T09:15:00Z,Bank B,SG,Alpha Mart,2026-03-07T10:00:00Z,QuickPay Settlement API
3,settled,M004,B002,S004,2100.0,2026-03-07T08:20:00Z,Bank C,US,Delta Travels,2026-03-07T10:00:00Z,QuickPay Settlement API
4,failed,M004,B002,S005,500.0,2026-03-07T08:50:00Z,Bank C,US,Delta Travels,2026-03-07T10:00:00Z,QuickPay Settlement API



Flat transaction table saved to api_normlized.csv
